# Preamble

In [1]:
# Running this script on a AWS Cloud SageMaker Notebook (ml.m5.2xlarge) with a volume size of 50 GB EBS

!python --version

Python 3.10.20


In [2]:
# Install Python packages that don't come pre-installed in AWS

!pip install scikit-learn

In [3]:
# Load required libraries

import boto3
import pandas as pd
import numpy as np
import re

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import Data

In [4]:
# Downloaded the CFPB compliants data from this website on June 16, 2026

# https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data

In [5]:
# Import the CSV file of complaints locally OR from the AWS S3 bucket

# https://aletheia-public.s3.us-east-2.amazonaws.com/complaints_16Jun2026.csv

load_data_locally = True

if load_data_locally:
    df = pd.read_csv("complaints_in_just_2025.csv")  # just data from 2025
else:
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026.csv')  # all years of data
    df = pd.read_csv(obj['Body'])

In [6]:
# The full dataset of complaints has 15,896,496 observations

df.shape

(5443005, 16)

In [7]:
# Below is a data dictionary of all columns in the CFPB compliants data
# https://cfpb.github.io/api/ccdb/fields.html

In [8]:
# List the raw data types of all columns of the Pandas data frame

df.dtypes

Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

In [9]:
# Convert the date columns from a string object format to a datetime format with just a date (i.e. YYYY-MM-DD)

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [10]:
# Look at a few obs

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2025-01-09,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account status incorrect,NaN,NaN,MOUNTAIN AMERICA FEDERAL CREDIT UNION,AZ,85301,NaN,Web,2025-01-27,Closed with explanation,Yes,11454251
1,2025-04-08,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,"To Whom It May Concern, I am filing a formal c...",NaN,MOHELA,NE,681XX,NaN,Web,2025-04-08,Closed with explanation,No,12856469
2,2025-04-15,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",Struggling with a XXXX has significantly impac...,NaN,"Maximus Federal Services, Inc.",AL,35007,NaN,Web,2026-04-06,Closed with explanation,Yes,12989708
3,2025-04-18,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Trouble with how payments are being handled,I have taken out parent loans on behalf of my ...,NaN,MOHELA,OH,43110,Servicemember,Web,2025-04-18,Closed with explanation,No,13062648
4,2025-05-05,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,NaN,NaN,MOHELA,OH,44224,NaN,Web,2025-05-05,Closed with explanation,No,13330457


In [11]:
# Group complaints by year received and count observations

df.groupby(df['Date received'].dt.year).size()

Date received
2025    5443005
dtype: int64

In [12]:
# Filter complaints in just the year 2025 in order to reduce the size of the Pandas data frame

df = df[df['Date received'].dt.year == 2025]

In [14]:
# Confirm the count of observations in year 2025

df.shape

(5443005, 16)

# Explore Data

In [15]:
# Group complaints by product category and count observations

df.groupby(df['Product']).size()

Product
Checking or savings account                                  84198
Credit card                                                  89989
Credit reporting or other personal consumer reports        4810314
Debt collection                                             283206
Debt or credit management                                     4267
Money transfer, virtual currency, or money service           84618
Mortgage                                                     24694
Payday loan, title loan, personal loan, or advance loan      13135
Prepaid card                                                  6499
Student loan                                                 21314
Vehicle loan or lease                                        20771
dtype: int64

In [16]:
# Convert the Product categorical variable into a collectino of dummy variables

df = pd.get_dummies(df, columns=['Product'], drop_first=True, dtype=int)

In [17]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

Issue
Advertising                                                        99
Advertising and marketing, including promotional offers          3106
Applying for a mortgage or refinancing an existing mortgage      2596
Attempts to collect debt not owed                              115683
Can't contact lender or servicer                                  203
                                                                ...  
Vehicle was repossessed or sold the vehicle                       149
Was approved for a loan, but didn't receive money                  23
Was approved for a loan, but didn't receive the money             105
Written notification about debt                                 64048
Wrong amount charged or received                                  459
Length: 90, dtype: int64

In [18]:
# Group by the tags and count obsevations

df.groupby(df['Tags']).size()

Tags
Older American                    29468
Older American, Servicemember      8004
Servicemember                    103326
dtype: int64

In [19]:
# Collapse the tags categorical feature into two separate binary features

df['Older American'] = np.where(df['Tags'].isin(['Older American','Older American, Servicemember']), 1, 0)
df['Servicemember'] = np.where(df['Tags'].isin(['Servicemember','Older American, Servicemember']), 1, 0)

In [20]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

Company public response
Company believes complaint caused principally by actions of third party outside the control or direction of the company        555
Company believes complaint is the result of an isolated error                                                                  378
Company believes complaint relates to a discontinued policy or procedure                                                         8
Company believes complaint represents an opportunity for improvement to better serve consumers                                 596
Company believes it acted appropriately as authorized by contract or law                                                     48420
Company believes the complaint is the result of a misunderstanding                                                             581
Company believes the complaint provided an opportunity to answer consumer's questions                                         2172
Company can't verify or dispute the facts in the complaint 

In [21]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

Company response to consumer
Closed with explanation            3194212
Closed with monetary relief          26109
Closed with non-monetary relief    2212222
In progress                            365
Untimely response                    10097
dtype: int64

In [22]:
# Collapse company response to consumer into a binary target: 

#   1 = complaint closed respectfully (with an explanation or a monetary relief)
#   0 = complaint closed with NO explanation

df['Target of Complaint Closed Respectfully'] = np.where(df['Company response to consumer'].isin(['Closed with explanation','Closed with monetary relief']), 1, 0)

In [23]:
# Group by target and count obsevations

df.groupby(df['Target of Complaint Closed Respectfully']).size()

Target of Complaint Closed Respectfully
0    2222684
1    3220321
dtype: int64

In [24]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

Timely response?
No       24595
Yes    5418410
dtype: int64

In [25]:
# Group by how complaint was submitted via and count obsevations

df.groupby(df['Submitted via']).size()

Submitted via
Phone            16421
Postal mail       7120
Referral         12187
Web            5407277
dtype: int64

In [26]:
# Convert the Product categorical variable into a collectino of dummy variables

df = pd.get_dummies(df, columns=['Submitted via'], drop_first=True, dtype=int)

In [27]:
# Count up the number of unique companies of consumer complaints

df['Company'].nunique()

3961

In [28]:
# Find the top 15 companies with the most consumer complaints

df['Company'].value_counts().head(15)

Company
EQUIFAX, INC.                             1680538
TRANSUNION INTERMEDIATE HOLDINGS, INC.    1601851
Experian Information Solutions Inc.       1388821
Block, Inc.                                 48091
CAPITAL ONE FINANCIAL CORPORATION           34102
CBC Companies, Inc.                         30769
Early Warning Services, LLC                 24509
JPMORGAN CHASE & CO.                        24458
Resurgent Capital Services L.P.             23741
CITIBANK, N.A.                              19791
LEXISNEXIS                                  19477
WELLS FARGO & COMPANY                       19468
BANK OF AMERICA, NATIONAL ASSOCIATION       19169
ENCORE CAPITAL GROUP INC.                   17404
NAVY FEDERAL CREDIT UNION                   17276
Name: count, dtype: int64

In [29]:
# Print consumer complaints where the complaint narrative contains a specific keyword

keyword = 'Data Science'

with pd.option_context('display.max_colwidth', None):
    filtered_column = df.loc[df['Consumer complaint narrative'].str.contains(keyword, case=False, na=False), 'Consumer complaint narrative']
    print(filtered_column)

1399533    I enrolled in a one-year Postgraduate in Data Science program with XXXX XXXX XXXX XXXX from XX/XX/XXXX to XX/XX/XXXX. Toward the end of the program, I received a phone call from a XXXX XXXX representative named XXXX, who offered me an upgrade to a XXXX XXXX from XXXX de XXXX, stating it was free to explore and that there would be no financial obligation unless I chose to proceed. \n\nXXXX sent me a link to a site called Kodakco.com, assuring me it was just to check the upgrade offer or try it temporarily. Trusting XXXX XXXX as an educational provider, I clicked the link and followed instructions but I was misled into believing this was a no-obligation trial, not a financial agreement. I now realize I XXXX have unintentionally agreed to a {$1000.00} Klarna financing under false pretenses. \n\nSoon after, I was notified by Klarna of a {$1000.00} debt in my name linked to XXXX. There was no transparent checkout process or consent to financing terms. I took immediate action cont

In [30]:
# Count up the number of actual complaint narratives: 1,221,970 out of 5,443,005 registered complaints in 2026

df[['Consumer complaint narrative']].count(axis=0)

Consumer complaint narrative    1221970
dtype: int64

# Feature Engineering

In [31]:
# Print out columns in the data frame

print(df.columns.tolist())

['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID', 'Product_Credit card', 'Product_Credit reporting or other personal consumer reports', 'Product_Debt collection', 'Product_Debt or credit management', 'Product_Money transfer, virtual currency, or money service', 'Product_Mortgage', 'Product_Payday loan, title loan, personal loan, or advance loan', 'Product_Prepaid card', 'Product_Student loan', 'Product_Vehicle loan or lease', 'Older American', 'Servicemember', 'Target of Complaint Closed Respectfully', 'Submitted via_Postal mail', 'Submitted via_Referral', 'Submitted via_Web']


In [32]:
# Count up the words in the consumer complaint narrative

df['complaint_word_count'] = df['Consumer complaint narrative'].str.split().str.len()

In [33]:
# Count up the X letters in the consumer complaint narrative

df['letter_X_count'] = df['Consumer complaint narrative'].str.count('X')

In [34]:
# Count up the $ dollar sign in the consumer complaint narrative

df['dollar_count'] = df['Consumer complaint narrative'].str.count('$')

In [35]:
# Count up keywords related to fraud

fraud_keywords = ['fraud', 'fraudulent', 'fake', 'forgery', 'trick', 'scam', 'swindle', 'hoax', 'scheme', 'sham']
fraud_pattern = '|'.join(fraud_keywords)

df['fraud_keyword'] = df['Consumer complaint narrative'].str.contains(fraud_pattern, case=False, na=False).astype(int)

In [36]:
# Convert NaN to 0

df = df.fillna(0)

# Prepare for Supervised Learning

In [37]:
# Subset just the relevant columns in the data frame

relevant_columns = ['Target of Complaint Closed Respectfully',
                     'Product_Credit card', 
                     'Product_Credit reporting or other personal consumer reports', 
                     'Product_Debt collection', 
                     'Product_Debt or credit management', 
                     'Product_Money transfer, virtual currency, or money service', 
                     'Product_Mortgage', 
                     'Product_Payday loan, title loan, personal loan, or advance loan', 
                     'Product_Prepaid card', 
                     'Product_Student loan', 
                     'Product_Vehicle loan or lease', 
                     'Older American', 
                     'Servicemember', 
                     'Submitted via_Postal mail', 
                     'Submitted via_Referral', 
                     'Submitted via_Web', 
                     'complaint_word_count', 
                     'letter_X_count', 
                     'dollar_count', 
                     'fraud_keyword']

subset_df = df[relevant_columns]

In [38]:
# Look at the data subset

subset_df.head()

,Target of Complaint Closed Respectfully,Product_Credit card,Product_Credit reporting or other personal consumer reports,Product_Debt collection,Product_Debt or credit management,"Product_Money transfer, virtual currency, or money service",Product_Mortgage,"Product_Payday loan, title loan, personal loan, or advance loan",Product_Prepaid card,Product_Student loan,Product_Vehicle loan or lease,Older American,Servicemember,Submitted via_Postal mail,Submitted via_Referral,Submitted via_Web,complaint_word_count,letter_X_count,dollar_count,fraud_keyword
0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0
1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,173.0,20.0,1.0,0
2,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,98.0,4.0,1.0,0
3,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,292.0,24.0,1.0,0
4,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0


In [39]:
# Separate feature vector (X) and target variable (y)

X = subset_df.drop('Target of Complaint Closed Respectfully', axis=1)
y = subset_df['Target of Complaint Closed Respectfully']

In [40]:
# Split the data into a training partition and a hold-out validation partition (70% training, 30% holdout)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Logistic Regression Model

In [41]:
# Scale features (Highly recommended for faster solver convergence) on just the Training partition to prevent data leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [42]:
# Initialize and fit the Logistic Regression model

logistic_reg_model = LogisticRegression(solver='lbfgs', max_iter=100)
logistic_reg_model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [43]:
# Make predictions

y_pred = logistic_reg_model.predict(X_test_scaled)          # Predicts binary class (0 or 1)
y_probs = logistic_reg_model.predict_proba(X_test_scaled)   # Predicts raw probabilities

In [44]:
# Evaluate the performance

print("Logsitic Regression Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Logsitic Regression Model Evaluation on Hold-out Data:
----------------------------------------
Accuracy Score: 0.59

Confusion Matrix:
[[   208 667793]
 [   211 964690]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.00      0.00    668001
           1       0.59      1.00      0.74    964901

    accuracy                           0.59   1632902
   macro avg       0.54      0.50      0.37   1632902
weighted avg       0.55      0.59      0.44   1632902



# Machine Learning Model

In [45]:
# Initialize and train the Gradient Boosting model on just the training partition

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42, max_depth=3)
gb_model.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [46]:
# Score and Evaluate on the hold-out validation partition

y_pred = gb_model.predict(X_test)

In [47]:
# Evaluate the performance

print("Machine Learning Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Machine Learning Model Evaluation on Hold-out Data:
----------------------------------------
Accuracy Score: 0.5910

Confusion Matrix:
[[   319 667682]
 [   222 964679]]

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.00      0.00    668001
           1       0.59      1.00      0.74    964901

    accuracy                           0.59   1632902
   macro avg       0.59      0.50      0.37   1632902
weighted avg       0.59      0.59      0.44   1632902

